In [ ]:
# 1. DEPENDÊNCIAS

from datetime import datetime
from decimal import Decimal

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [ ]:
# 2. CONFIGURAÇÕES

SOURCE_TABLE = "Silver.caged"
SALARY_MIN_TABLE = "Silver.salario_minimo"
SILVER_CONTROL_TABLE = "Silver.control_caged_quality"

GOLD_SCHEMA = "Gold"
GOLD_FACT_TABLE = f"{GOLD_SCHEMA}.fato_caged"
GOLD_CONTROL_TABLE = f"{GOLD_SCHEMA}.control_caged_quality"

MIN_SALARY_RATIO = 0.3
MAX_SALARY_RATIO = 150.0

# Nomes canônicos usados internamente e na Gold.
COLUMN_ALIASES = {
    "competencia_mov": [
        "competencia_mov",
        "competência_mov",
        "competênciamov",
        "competenciamov",
    ],
    "municipio": [
        "municipio",
        "município",
    ],
    "subclasse": [
        "subclasse",
    ],
    "cbo2002ocupacao": [
        "cbo2002ocupacao",
        "cbo2002ocupação",
    ],
    "sexo": [
        "sexo",
    ],
    "racacor": [
        "racacor",
        "raçacor",
    ],
    "grau_instrucao": [
        "grau_instrucao",
        "graudeinstrução",
        "graudeinstrucao",
    ],
    "idade": [
        "idade",
    ],
    "salario": [
        "salario",
        "salário",
    ],
    "saldomovimentacao": [
        "saldomovimentacao",
        "saldomovimentação",
    ],
    "indtrabintermitente": [
        "indtrabintermitente",
    ],
}

SALARY_MIN_ALIASES = {
    "data": ["data"],
    "salario_min": ["salario_min", "saláriomin", "salario_minimo"],
}

CONTROL_ALIASES = {
    "competencia_mov": [
        "competencia_mov",
        "competência_mov",
        "competênciamov",
        "competenciamov",
    ],
    "status": ["status"],
    "metric_group": ["metric_group"],
    "metric_name": ["metric_name"],
}

In [ ]:
# 3. FUNÇÕES AUXILIARES

def now_utc():
    """
    Retorna timestamp UTC sem tzinfo para compatibilidade com Spark TIMESTAMP.
    """
    return datetime.utcnow()


def table_exists(table_name: str) -> bool:
    """
    Verifica se uma tabela Spark/Delta existe.
    """
    return spark.catalog.tableExists(table_name)


def first_existing_column(df, aliases, required=True):
    """
    Retorna a primeira coluna existente dentre os aliases informados.
    """
    columns = set(df.columns)
    for name in aliases:
        if name in columns:
            return name

    if required:
        raise ValueError(
            f"Nenhuma das colunas esperadas foi encontrada. "
            f"Esperadas={aliases}. Colunas disponíveis={df.columns}"
        )

    return None


def normalize_caged(df):
    """
    Padroniza as colunas necessárias da Silver para nomes canônicos.
    """
    selected = []

    for canonical_name, aliases in COLUMN_ALIASES.items():
        source_name = first_existing_column(df, aliases, required=True)

        if canonical_name == "competencia_mov":
            selected.append(
                F.to_date(F.col(source_name)).alias(canonical_name)
            )

        elif canonical_name == "salario":
            selected.append(
                F.col(source_name).cast("double").alias(canonical_name)
            )

        else:
            selected.append(F.col(source_name).alias(canonical_name))

    return df.select(*selected)


def normalize_salary_min(df):
    """
    Padroniza a tabela de salário mínimo.
    """
    data_col = first_existing_column(
        df,
        SALARY_MIN_ALIASES["data"],
        required=True
    )

    salary_col = first_existing_column(
        df,
        SALARY_MIN_ALIASES["salario_min"],
        required=True
    )

    return (
        df.select(
            F.to_date(F.col(data_col)).alias("data"),
            F.col(salary_col).cast("double").alias("salario_min"),
        )
    )


def append_control_rows(rows):
    """
    Acrescenta linhas à tabela de controle Gold.
    """
    if not rows:
        return

    schema = T.StructType([
        T.StructField("execution_id", T.StringType(), True),
        T.StructField("source_table", T.StringType(), True),
        T.StructField("target_table", T.StringType(), True),
        T.StructField("competencia_mov", T.DateType(), True),
        T.StructField("execution_start", T.TimestampType(), True),
        T.StructField("processed_at", T.TimestampType(), True),
        T.StructField("status", T.StringType(), True),
        T.StructField("metric_group", T.StringType(), True),
        T.StructField("metric_name", T.StringType(), True),
        T.StructField("column_name", T.StringType(), True),
        T.StructField("metric_value", T.DecimalType(38, 10), True),
        T.StructField("metric_unit", T.StringType(), True),
        T.StructField("metric_status", T.StringType(), True),
        T.StructField("details", T.StringType(), True),
        T.StructField("error_message", T.StringType(), True),
    ])

    normalized_rows = []

    for row in rows:
        value = row.get("metric_value")

        if value is not None:
            value = Decimal(str(value))

        normalized_rows.append((
            row.get("execution_id"),
            row.get("source_table"),
            row.get("target_table"),
            row.get("competencia_mov"),
            row.get("execution_start"),
            row.get("processed_at"),
            row.get("status"),
            row.get("metric_group"),
            row.get("metric_name"),
            row.get("column_name"),
            value,
            row.get("metric_unit"),
            row.get("metric_status"),
            row.get("details"),
            row.get("error_message"),
        ))

    (
        spark.createDataFrame(normalized_rows, schema=schema)
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(GOLD_CONTROL_TABLE)
    )


def add_metric(
    rows,
    execution_id,
    execution_start,
    competencia,
    status,
    metric_group,
    metric_name,
    metric_value=None,
    column_name=None,
    metric_unit=None,
    metric_status="INFO",
    details=None,
    error_message=None,
    processed_at=None,
):
    """
    Adiciona uma métrica ao buffer do controle.
    """
    rows.append({
        "execution_id": execution_id,
        "source_table": SOURCE_TABLE,
        "target_table": GOLD_FACT_TABLE,
        "competencia_mov": competencia,
        "execution_start": execution_start,
        "processed_at": processed_at,
        "status": status,
        "metric_group": metric_group,
        "metric_name": metric_name,
        "column_name": column_name,
        "metric_value": metric_value,
        "metric_unit": metric_unit,
        "metric_status": metric_status,
        "details": details,
        "error_message": error_message,
    })


def get_silver_success_competencies(silver_control_df):
    """
    Descobre as competências carregadas com sucesso na Silver.

    A tabela de controle mostrada no modelo possui:
    status, metric_group, metric_name e competencia_mov.
    Quando metric_name existe, priorizamos o registro de load_success.
    """
    comp_col = first_existing_column(
        silver_control_df,
        CONTROL_ALIASES["competencia_mov"],
        required=True,
    )
    status_col = first_existing_column(
        silver_control_df,
        CONTROL_ALIASES["status"],
        required=True,
    )
    group_col = first_existing_column(
        silver_control_df,
        CONTROL_ALIASES["metric_group"],
        required=False,
    )
    metric_col = first_existing_column(
        silver_control_df,
        CONTROL_ALIASES["metric_name"],
        required=False,
    )

    df = silver_control_df.withColumn(
        "__competencia_mov",
        F.to_date(F.col(comp_col))
    )

    df = df.filter(F.upper(F.col(status_col)) == "SUCCESS")

    if group_col is not None:
        df = df.filter(F.lower(F.col(group_col)) == "execution")

    if metric_col is not None:
        # Compatível com o padrão mostrado no controle Silver.
        load_success = df.filter(
            F.lower(F.col(metric_col)).isin(
                "load_success",
                "loadsuccess",
                "silver_success"
            )
        )

        if load_success.limit(1).count() > 0:
            df = load_success

    return (
        df.select("__competencia_mov")
        .where(F.col("__competencia_mov").isNotNull())
        .distinct()
        .withColumnRenamed("__competencia_mov", "competencia_mov")
    )


def get_gold_success_competencies(gold_control_df):
    """
    Retorna competências que já tiveram agregação Gold concluída com sucesso.
    """
    if gold_control_df is None:
        return spark.createDataFrame(
            [],
            T.StructType([
                T.StructField("competencia_mov", T.DateType(), True)
            ])
        )

    comp_col = first_existing_column(
        gold_control_df,
        CONTROL_ALIASES["competencia_mov"],
        required=True,
    )
    status_col = first_existing_column(
        gold_control_df,
        CONTROL_ALIASES["status"],
        required=True,
    )
    group_col = first_existing_column(
        gold_control_df,
        CONTROL_ALIASES["metric_group"],
        required=False,
    )
    metric_col = first_existing_column(
        gold_control_df,
        CONTROL_ALIASES["metric_name"],
        required=False,
    )

    df = gold_control_df.withColumn(
        "__competencia_mov",
        F.to_date(F.col(comp_col))
    )

    df = df.filter(F.upper(F.col(status_col)) == "SUCCESS")

    if group_col is not None:
        df = df.filter(F.lower(F.col(group_col)) == "execution")

    if metric_col is not None:
        df = df.filter(
            F.lower(F.col(metric_col)).isin(
                "aggregation_success",
                "gold_success"
            )
        )

    return (
        df.select("__competencia_mov")
        .where(F.col("__competencia_mov").isNotNull())
        .distinct()
        .withColumnRenamed("__competencia_mov", "competencia_mov")
    )


def create_gold_tables_if_needed(normalized_caged_df):
    """
    Cria schema, fato e controle Gold quando necessário.
    """
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_SCHEMA}")

    # Fato Gold.
    if not table_exists(GOLD_FACT_TABLE):
        empty_gold = (
            normalized_caged_df
            .limit(0)
            .select(
                "competencia_mov",
                "municipio",
                "subclasse",
                "cbo2002ocupacao",
                "sexo",
                "racacor",
                "grau_instrucao",
                "idade",
            )
            .withColumn("qtd_movimentacoes", F.lit(0).cast("long"))
            .withColumn("qtd_admissoes", F.lit(0).cast("long"))
            .withColumn("qtd_desligamentos", F.lit(0).cast("long"))
            .withColumn("salario", F.lit(0).cast("decimal(18,2)"))
            .withColumn("qtd_salarios", F.lit(0).cast("long"))
        )

        (
            empty_gold.write
            .format("delta")
            .mode("overwrite")
            .partitionBy("competencia_mov")
            .saveAsTable(GOLD_FACT_TABLE)
        )

    # Controle/qualidade Gold.
    if not table_exists(GOLD_CONTROL_TABLE):
        control_schema = T.StructType([
            T.StructField("execution_id", T.StringType(), True),
            T.StructField("source_table", T.StringType(), True),
            T.StructField("target_table", T.StringType(), True),
            T.StructField("competencia_mov", T.DateType(), True),
            T.StructField("execution_start", T.TimestampType(), True),
            T.StructField("processed_at", T.TimestampType(), True),
            T.StructField("status", T.StringType(), True),
            T.StructField("metric_group", T.StringType(), True),
            T.StructField("metric_name", T.StringType(), True),
            T.StructField("column_name", T.StringType(), True),
            T.StructField("metric_value", T.DecimalType(38, 10), True),
            T.StructField("metric_unit", T.StringType(), True),
            T.StructField("metric_status", T.StringType(), True),
            T.StructField("details", T.StringType(), True),
            T.StructField("error_message", T.StringType(), True),
        ])

        empty_control = spark.createDataFrame([], control_schema)

        (
            empty_control.write
            .format("delta")
            .mode("overwrite")
            .partitionBy("competencia_mov")
            .saveAsTable(GOLD_CONTROL_TABLE)
        )


def calculate_null_metrics(month_df, competencia, execution_id, execution_start, control_rows):
    """
    Calcula quantidade e percentual de nulos para TODAS as colunas da Silver.
    """
    aliases = {
        "__rows": F.count(F.lit(1)),
    }

    for idx, col_name in enumerate(month_df.columns):
        aliases[f"__null_{idx}"] = F.sum(
            F.when(F.col(col_name).isNull(), 1).otherwise(0)
        )

    result = month_df.agg(*[
        expr.alias(alias)
        for alias, expr in aliases.items()
    ]).first()

    rows_input = int(result["__rows"] or 0)

    add_metric(
        control_rows,
        execution_id,
        execution_start,
        competencia,
        "RUNNING",
        "execution",
        "rows_input",
        rows_input,
        metric_unit="rows",
        metric_status="INFO",
    )

    for idx, col_name in enumerate(month_df.columns):
        null_count = int(result[f"__null_{idx}"] or 0)
        null_pct = (null_count / rows_input * 100.0) if rows_input else 0.0

        status = "OK" if null_count == 0 else "INFO"

        add_metric(
            control_rows,
            execution_id,
            execution_start,
            competencia,
            "RUNNING",
            "quality",
            "null_count",
            null_count,
            column_name=col_name,
            metric_unit="rows",
            metric_status=status,
        )

        add_metric(
            control_rows,
            execution_id,
            execution_start,
            competencia,
            "RUNNING",
            "quality",
            "null_pct",
            null_pct,
            column_name=col_name,
            metric_unit="percent",
            metric_status=status,
        )

    return rows_input


def calculate_salary_metrics(
    month_df,
    competencia,
    execution_id,
    execution_start,
    control_rows,
):
    """
    Estatísticas do salário da Silver antes do filtro:
    min, max, média e mediana.
    """
    stats = (
        month_df
        .agg(
            F.min("salario").alias("min"),
            F.max("salario").alias("max"),
            F.avg("salario").alias("avg"),
            F.expr(
                "percentile_approx(salario, 0.5, 10000)"
            ).alias("median"),
        )
        .first()
    )

    for metric_name, value in [
        ("salary_min", stats["min"]),
        ("salary_max", stats["max"]),
        ("salary_avg", stats["avg"]),
        ("salary_median", stats["median"]),
    ]:
        add_metric(
            control_rows,
            execution_id,
            execution_start,
            competencia,
            "RUNNING",
            "quality",
            metric_name,
            value,
            column_name="salario",
            metric_unit="BRL",
            metric_status="OK" if value is not None else "INFO",
        )


def merge_gold_month(gold_month_df, competencia):
    """
    MERGE idempotente da competência.

    A chave lógica é exatamente a granularidade do groupBy.
    Reexecutar a competência atualiza a mesma combinação de chaves
    em vez de inserir duplicidades.
    """
    if gold_month_df.limit(1).count() == 0:
        return 0

    target = DeltaTable.forName(spark, GOLD_FACT_TABLE)

    key_columns = [
        "competencia_mov",
        "municipio",
        "subclasse",
        "cbo2002ocupacao",
        "sexo",
        "racacor",
        "grau_instrucao",
        "idade",
    ]

    conditions = [
        f"t.{col_name} <=> s.{col_name}"
        for col_name in key_columns
    ]

    merge_condition = " AND ".join(conditions)

    (
        target.alias("t")
        .merge(
            gold_month_df.alias("s"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    return gold_month_df.count()


def process_competencia(
    competencia,
    caged_df,
    salary_min_df,
    caged_raw_df,
    raw_competencia_col,
):
    """
    Processa uma única competência mensal.
    """
    execution_id = str(__import__("uuid").uuid4())
    execution_start = now_utc()
    control_rows = []

    add_metric(
        control_rows,
        execution_id,
        execution_start,
        competencia,
        "RUNNING",
        "execution",
        "aggregation_start",
        1,
        metric_unit="flag",
        metric_status="INFO",
        details="Início do processamento mensal da Gold."
    )

    # 1. Carrega somente a competência.

    # Mantém todas as colunas originais para a auditoria de qualidade.
    month_raw_df = caged_raw_df.filter(
        F.to_date(F.col(raw_competencia_col)) == F.lit(competencia)
    ).cache()

    rows_input = calculate_null_metrics(
        month_raw_df,
        competencia,
        execution_id,
        execution_start,
        control_rows,
    )

    if rows_input == 0:
        processed_at = now_utc()

        add_metric(
            control_rows,
            execution_id,
            execution_start,
            competencia,
            "ERROR",
            "execution",
            "aggregation_error",
            None,
            metric_unit="flag",
            metric_status="ERROR",
            details="A competência está no controle Silver, mas não possui linhas na Silver.caged.",
            error_message="Silver.caged sem registros para a competência.",
            processed_at=processed_at,
        )

        append_control_rows(control_rows)
        month_raw_df.unpersist()

        return {
            "competencia_mov": competencia,
            "status": "ERROR",
            "error": "Silver.caged sem registros para a competência.",
            "execution_id": execution_id,
        }

    # Padroniza somente as colunas necessárias para transformação/agregação.
    month_df = normalize_caged(month_raw_df).cache()

    calculate_salary_metrics(
        month_df,
        competencia,
        execution_id,
        execution_start,
        control_rows,
    )

    # 2. Salário mínimo da competência.

    salary_min_month = (
        salary_min_df
        .filter(F.col("data") == F.lit(competencia))
        .select("data", "salario_min")
    )

    salary_min_count = salary_min_month.count()

    if salary_min_count == 0:
        processed_at = now_utc()

        add_metric(
            control_rows,
            execution_id,
            execution_start,
            competencia,
            "ERROR",
            "quality",
            "salary_min_missing",
            1,
            column_name="salario_min",
            metric_unit="flag",
            metric_status="ERROR",
            details="Não existe salário mínimo correspondente à competência.",
            error_message="Silver.salario_minimo sem registro para a competência.",
            processed_at=processed_at,
        )

        append_control_rows(control_rows)
        month_df.unpersist()
        month_raw_df.unpersist()

        return {
            "competencia_mov": competencia,
            "status": "ERROR",
            "error": "Salário mínimo ausente para a competência.",
            "execution_id": execution_id,
        }

    salary_min_duplicates = (
        salary_min_month
        .groupBy("data")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    if salary_min_duplicates > 0:
        processed_at = now_utc()

        add_metric(
            control_rows,
            execution_id,
            execution_start,
            competencia,
            "ERROR",
            "quality",
            "salary_min_reference_inconsistency",
            salary_min_duplicates,
            column_name="data",
            metric_unit="rows",
            metric_status="ERROR",
            details="Existem múltiplos registros de salário mínimo para a mesma competência.",
            error_message="Referência Silver.salario_minimo não é unívoca por competência.",
            processed_at=processed_at,
        )

        append_control_rows(control_rows)
        month_df.unpersist()
        month_raw_df.unpersist()

        return {
            "competencia_mov": competencia,
            "status": "ERROR",
            "error": "Salário mínimo duplicado para a competência.",
            "execution_id": execution_id,
        }


    # 3. Join 1:1 com salário mínimo.
    #    LEFT JOIN para podermos medir inconsistências.


    enriched_df = (
        month_df.alias("c")
        .join(
            salary_min_month.alias("sm"),
            F.col("c.competencia_mov") == F.col("sm.data"),
            "left",
        )
        .select(
            "c.*",
            F.col("sm.salario_min").alias("salario_min"),
        )
        .withColumn(
            "razao_salario_sm",
            F.when(
                F.col("salario_min") != 0,
                F.col("salario") / F.col("salario_min")
            )
        )
    )

    # 4. Métricas de inconsistência/filtro.

    salary_null_expr = F.col("salario").isNull()
    salary_min_null_expr = F.col("salario_min").isNull()

    salary_outlier_expr = (
        F.col("salario").isNotNull()
        & F.col("salario_min").isNotNull()
        & (
            (F.col("razao_salario_sm") < F.lit(MIN_SALARY_RATIO))
            | (F.col("razao_salario_sm") > F.lit(MAX_SALARY_RATIO))
        )
    )

    valid_expr = (
        F.col("salario").isNotNull()
        & F.col("salario_min").isNotNull()
        & F.col("razao_salario_sm").between(
            F.lit(MIN_SALARY_RATIO),
            F.lit(MAX_SALARY_RATIO)
        )
    )

    filter_stats = (
        enriched_df
        .agg(
            F.count("*").alias("rows_after_join"),
            F.sum(F.when(salary_null_expr, 1).otherwise(0)).alias("salary_null"),
            F.sum(F.when(salary_min_null_expr, 1).otherwise(0)).alias("salary_min_null"),
            F.sum(F.when(salary_outlier_expr, 1).otherwise(0)).alias("salary_outlier"),
            F.sum(F.when(valid_expr, 1).otherwise(0)).alias("valid_rows"),
        )
        .first()
    )

    rows_after_join = int(filter_stats["rows_after_join"] or 0)
    salary_null_count = int(filter_stats["salary_null"] or 0)
    salary_min_null_count = int(filter_stats["salary_min_null"] or 0)
    salary_outlier_count = int(filter_stats["salary_outlier"] or 0)
    valid_rows = int(filter_stats["valid_rows"] or 0)
    discarded_rows = rows_after_join - valid_rows

    add_metric(
        control_rows,
        execution_id,
        execution_start,
        competencia,
        "RUNNING",
        "quality",
        "inconsistency_count",
        salary_null_count,
        column_name="salario",
        metric_unit="rows",
        metric_status="INFO" if salary_null_count else "OK",
        details="Salário nulo.",
    )

    add_metric(
        control_rows,
        execution_id,
        execution_start,
        competencia,
        "RUNNING",
        "quality",
        "inconsistency_count",
        salary_min_null_count,
        column_name="salario_min",
        metric_unit="rows",
        metric_status="INFO" if salary_min_null_count else "OK",
        details="Competência sem salário mínimo correspondente.",
    )

    add_metric(
        control_rows,
        execution_id,
        execution_start,
        competencia,
        "RUNNING",
        "quality",
        "inconsistency_count",
        salary_outlier_count,
        column_name="salario",
        metric_unit="rows",
        metric_status="INFO" if salary_outlier_count else "OK",
        details=(
            f"Salário fora da faixa de "
            f"{MIN_SALARY_RATIO} a {MAX_SALARY_RATIO} salários mínimos."
        ),
    )

    add_metric(
        control_rows,
        execution_id,
        execution_start,
        competencia,
        "RUNNING",
        "transformation",
        "rows_discarded",
        discarded_rows,
        metric_unit="rows",
        metric_status="INFO" if discarded_rows else "OK",
        details="Linhas removidas antes da agregação por regra de salário."
    )

    discarded_pct = (
        discarded_rows / rows_after_join * 100.0
        if rows_after_join
        else 0.0
    )

    add_metric(
        control_rows,
        execution_id,
        execution_start,
        competencia,
        "RUNNING",
        "transformation",
        "rows_discarded_pct",
        discarded_pct,
        metric_unit="percent",
        metric_status="INFO" if discarded_rows else "OK",
    )

    add_metric(
        control_rows,
        execution_id,
        execution_start,
        competencia,
        "RUNNING",
        "transformation",
        "rows_processed",
        valid_rows,
        metric_unit="rows",
        metric_status="OK" if valid_rows else "INFO",
    )

    # 5. Filtra a população válida.

    valid_df = (
        enriched_df
        .filter(valid_expr)
        .drop("salario_min", "razao_salario_sm")
    )

    # 6. Agregação Gold.

    group_columns = [
        "competencia_mov",
        "municipio",
        "subclasse",
        "cbo2002ocupacao",
        "sexo",
        "racacor",
        "grau_instrucao",
        "idade",
    ]

    gold_month_df = (
        valid_df
        .groupBy(*group_columns)
        .agg(
            F.count("*").alias("qtd_movimentacoes"),

            F.sum(
                F.when(
                    F.col("saldomovimentacao") == 1,
                    1
                ).otherwise(0)
            ).cast("long").alias("qtd_admissoes"),

            F.sum(
                F.when(
                    F.col("saldomovimentacao") == -1,
                    1
                ).otherwise(0)
            ).cast("long").alias("qtd_desligamentos"),

            F.sum(
                F.when(
                    (F.col("saldomovimentacao") == 1)
                    & (F.col("indtrabintermitente") == 0),
                    F.col("salario").cast("decimal(18,2)")
                ).otherwise(
                    F.lit(0).cast("decimal(18,2)")
                )
            ).cast("decimal(18,2)").alias("salario"),

            F.sum(
                F.when(
                    (F.col("saldomovimentacao") == 1)
                    & (F.col("indtrabintermitente") == 0),
                    1
                ).otherwise(0)
            ).cast("long").alias("qtd_salarios"),
        )
    )

    gold_rows = gold_month_df.count()

    add_metric(
        control_rows,
        execution_id,
        execution_start,
        competencia,
        "RUNNING",
        "transformation",
        "gold_rows_generated",
        gold_rows,
        metric_unit="rows",
        metric_status="OK" if gold_rows else "INFO",
    )

    # 7. MERGE idempotente.

    merged_rows = merge_gold_month(
        gold_month_df,
        competencia
    )

    add_metric(
        control_rows,
        execution_id,
        execution_start,
        competencia,
        "RUNNING",
        "execution",
        "merge_rows_source",
        merged_rows,
        metric_unit="rows",
        metric_status="OK",
    )

    # 8. Métricas da população válida.

    valid_salary_stats = (
        valid_df
        .agg(
            F.min("salario").alias("min"),
            F.max("salario").alias("max"),
            F.avg("salario").alias("avg"),
            F.expr(
                "percentile_approx(salario, 0.5, 10000)"
            ).alias("median"),
        )
        .first()
    )

    for metric_name, value in [
        ("valid_salary_min", valid_salary_stats["min"]),
        ("valid_salary_max", valid_salary_stats["max"]),
        ("valid_salary_avg", valid_salary_stats["avg"]),
        ("valid_salary_median", valid_salary_stats["median"]),
    ]:
        add_metric(
            control_rows,
            execution_id,
            execution_start,
            competencia,
            "RUNNING",
            "quality",
            metric_name,
            value,
            column_name="salario",
            metric_unit="BRL",
            metric_status="OK" if value is not None else "INFO",
        )

    # 9. Finaliza sucesso.

    processed_at = now_utc()

    add_metric(
        control_rows,
        execution_id,
        execution_start,
        competencia,
        "SUCCESS",
        "execution",
        "aggregation_success",
        1,
        metric_unit="flag",
        metric_status="OK",
        details=(
            f"Competência {competencia} processada na Gold com sucesso."
        ),
        processed_at=processed_at,
    )

    duration_seconds = (
        processed_at - execution_start
    ).total_seconds()

    add_metric(
        control_rows,
        execution_id,
        execution_start,
        competencia,
        "SUCCESS",
        "execution",
        "execution_duration",
        duration_seconds,
        metric_unit="seconds",
        metric_status="INFO",
        processed_at=processed_at,
    )

    append_control_rows(control_rows)

    month_df.unpersist()
    month_raw_df.unpersist()

    return {
        "competencia_mov": competencia,
        "status": "SUCCESS",
        "rows_input": rows_input,
        "rows_discarded": discarded_rows,
        "rows_processed": valid_rows,
        "gold_rows": gold_rows,
        "execution_id": execution_id,
    }

In [ ]:
# 4. LEITURA INICIAL

caged_raw = spark.table(SOURCE_TABLE)
salary_min_raw = spark.table(SALARY_MIN_TABLE)
silver_control = spark.table(SILVER_CONTROL_TABLE)

raw_competencia_col = first_existing_column(
    caged_raw,
    COLUMN_ALIASES["competencia_mov"],
    required=True,
)

caged = normalize_caged(caged_raw).cache()
salary_min = normalize_salary_min(salary_min_raw)

In [ ]:
# 5. CRIA SCHEMA E TABELAS GOLD

create_gold_tables_if_needed(caged)

In [ ]:
# 6. DESCOBRE COMPETÊNCIAS

silver_competencies = get_silver_success_competencies(
    silver_control
)

gold_control = spark.table(GOLD_CONTROL_TABLE)

gold_success_competencies = get_gold_success_competencies(
    gold_control
)

pending_competencies_df = (
    silver_competencies
    .join(
        gold_success_competencies,
        on="competencia_mov",
        how="left_anti",
    )
    .orderBy("competencia_mov")
)

In [ ]:
# 7. LISTA DE COMPETÊNCIAS PENDENTES

pending_competencies = [
    row["competencia_mov"]
    for row in pending_competencies_df.collect()
]

print(
    f"Competências Silver elegíveis: "
    f"{silver_competencies.count()}"
)

print(
    f"Competências Gold já processadas: "
    f"{gold_success_competencies.count()}"
)

print(
    f"Competências pendentes: "
    f"{len(pending_competencies)}"
)

if pending_competencies:
    print(
        "Pendentes: "
        + ", ".join(str(x) for x in pending_competencies)
    )
else:
    print("Nenhuma competência nova para processar.")

In [ ]:
# 8. PROCESSAMENTO MENSAL


results = []

for competencia in pending_competencies:

    print("=" * 70)
    print(f"PROCESSANDO COMPETÊNCIA: {competencia}")
    print("=" * 70)

    try:
        result = process_competencia(
            competencia=competencia,
            caged_df=caged,
            salary_min_df=salary_min,
            caged_raw_df=caged_raw,
            raw_competencia_col=raw_competencia_col,
        )

        results.append(result)

        if result["status"] == "SUCCESS":
            print(
                f"SUCCESS | {competencia} | "
                f"entrada={result['rows_input']} | "
                f"descartadas={result['rows_discarded']} | "
                f"processadas={result['rows_processed']} | "
                f"gold={result['gold_rows']}"
            )
        else:
            print(
                f"ERROR | {competencia} | "
                f"{result.get('error', 'Erro no processamento.')}"
            )

    except Exception as exc:
        # A função já registra ERROR nos casos previstos.
        # Este bloco registra falhas não previstas.
        execution_id = str(__import__("uuid").uuid4())
        execution_start = now_utc()
        processed_at = now_utc()

        error_rows = []

        add_metric(
            error_rows,
            execution_id,
            execution_start,
            competencia,
            "ERROR",
            "execution",
            "aggregation_error",
            None,
            metric_unit="flag",
            metric_status="ERROR",
            details=f"Falha não prevista ao processar {competencia}.",
            error_message=str(exc),
            processed_at=processed_at,
        )

        append_control_rows(error_rows)

        results.append({
            "competencia_mov": competencia,
            "status": "ERROR",
            "error": str(exc),
        })

        print(
            f"ERROR | {competencia} | {exc}"
        )

In [ ]:
# 9. LIMPEZA


caged.unpersist()

In [ ]:
# 10. RESUMO FINAL


print("=" * 70)
print("RESUMO DO PROCESSAMENTO GOLD")
print("=" * 70)

for result in results:
    print(result)